# 7.3. Padding and Stride
D2L의 Padding and Stride장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [2]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 합성곱을 하면 이미지가 작아진다.

입력이 3 x 3이고 커널이 2 x 2라면 출력은 2 x 2이다.

$$
(n_h​−k_h​+1)×(n_w​−k_w​+1)
$$

예를 들어 입력이 8 x 8이고 커널이 3 x 3이면 8 - 3 + 1 = 6 이기 때문에 6 x 6이다.

여러 층에서 반복되면 feature map의 크기가 계속 작아진다. D2L에선 이를 방지하거나 반대로 의도적으로 크기를 줄이기 위해 padding과 stride를 사용한다.

## 2. Padding이 필요한 이유

CNN에서는 보통 커널이 이미지 위를 움직인다. 그런데 이미지 가운데 픽셀은 여러 번 커널에 포함되지만, 가장자리 픽셀은 상대적으로 적게 사용된다.

```text
X X X X X
X O O O X
X O O O X
X O O O X
X X X X X
```

가운데 O 픽셀은 여러 convolution 연산에 참여하지만 가장자리 X 픽셀은 상대적으로 적게 참여한다. 

convolution을 할 때마다 이미지도 줄어든다. 그래서 입력 이미지 주변을 인위적으로 채우는 방법이 padding이다.

## 3. Padding이란?

가장 흔한 방법은 이미지 주변에 0을 추가하는 것이다. 이를 zero padding이라고 한다.

만약에 입력이
```text
1 2 3
4 5 6
7 8 9

padding = 1을 적용하면
0 0 0 0 0
0 1 2 3 0
0 4 5 6 0
0 7 8 9 0
0 0 0 0 0
```

3 x 3 이었던게 입력이 5 x 5 처럼 커졌다. padding = 1은 PyTorch에서 위, 아래, 왼쪽, 오른쪽에 각각 1칸씩 추가한다는 의미이다.

## 4. Padding으로 입력과 출력 크기를 같게 만들기

CNN에서 가장 많이 쓰는 조합은 이렇다.

```text
kernel_size = 3
padding = 1
stride = 1
```

예를 들어 입력이 8 x 8 이면 padding 이후 영역은 10 x 10이 된다. 그 상태에서 3 x 3커널로 convolution하면 8 x 8이 된다.

출력 크기가 유지된다. 그래서 CNN에서 3 x 3커널과 padding = 1조합을 자주보게 된다.

## 5. Padding 출력 크기 공식

D2L에서는 p를 양쪽 padding의 총합으로 정의한다.

그래서 이렇게 되는데
$$
n−k+p+1
$$

PyTorch에서 padding = 1이라고 하면 양쪽에 1개씩 들어간다. 그래서 실전에선 이런 공식을 더 많이 사용한다고 한다.

$$
output=n+2p−k+1
$$

```text
n = 입력 크기
k = kernel 크기
p = 한쪽에 넣는 padding
```

## 6. PyTorch에서 Padding 사용하기

In [ ]:
X = torch.rand(1, 1, 8, 8) # batch size, channels, height, width

print(X.shape)

torch.Size([1, 1, 8, 8])


In [ ]:
conv = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    padding=1
)

Y = conv(X) # 3 x 3 convolution 적용

print(Y.shape) # 크기가 유지된다.

torch.Size([1, 1, 8, 8])


## 7. 커널 크기는 보통 홀수로 사용할까?

CNN에선 보통 홀수 커널 크기를 자주 사용한다. 이유는 padding을 대칭적으로 넣기 쉽기 때문이다.

예를 들어서 kernel = 3이라면 padding = 1을 양쪽에 넣으면 된다.

    0 | X X X X | 0

5 x 5커널이라면 padding = 2로 하면 된다.

일반적으로 홀수 커널에서 

$$
padding = \frac{kernel\ size - 1}{2}
$$

이렇게 두면 stride가 1일 때 입력과 출력 크기를 동일하게 유지하기 쉽다.


## 8. Stride란?

지금까지 커널은 한 번 게산하고 옆으로 1칸씩 이동했다. 이게 stride = 1이다.

stride가 2라면 두칸씩 이동한다. 커널이 한 번 convolution 한 뒤 다음 위치로 몇 칸 이동할지를 결정하는 값이다.

## 9. Stride가 커지면 출력이 작아진다.

```text
입력 = 8 × 8
kernel = 3 × 3
padding = 1
stride = 1

라면 크기가 그대로 유지된다.
```

stride가 2라면 커널이 두 칸씩 건나뛰며 움직인다. 계산하는 위치 자체가 줄어든다.

```text
8 × 8
   ↓
kernel=3
padding=1
stride=2
   ↓
4 × 4
```

## 10. Padding + Stride 출력 크기 공식

$$
\left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1
$$

```text
n = 입력 크기
k = kernel 크기
p = 한쪽 padding 크기
s = stride
```

floor는 소수점 이하를 버린다는 뜻이다.

예를 들어서 입력 = 8 kernel = 3 padding = 1 stride = 2 라면

$$
\left\lfloor \frac{8 + 2 - 3}{2} \right\rfloor + 1\\ = \lfloor3.5\rfloor + 1\\ = 3 + 1 = 4
$$

## 11. PyTorch에서 Stride 확인하기

In [5]:
conv = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    padding=1,
    stride=2
)

X = torch.rand(1, 1, 8, 8)

Y = conv(X)

print(X.shape)
print(Y.shape)

torch.Size([1, 1, 8, 8])
torch.Size([1, 1, 4, 4])


## 12. Padding과 Stride의 역할 구분

`kernel_size` = 한 번에 얼마만큼 볼 것인가?  
`padding`     = 가장자리를 얼마나 채울 것인가?  
`stride`      = 커널을 몇 칸씩 이동할 것인가?  

예를 들어 이렇게 되어있다면
```py
nn.Conv2d(
    1,
    16, # 16개의 feature map 만들기
    kernel_size=3, # 3 x 3영역을 보기
    padding=1, # 가장자리 0을 1칸씩 추가
    stride=2 # 커널을 2칸씩 이동하면서
)
```

## 13. 실제 CNN에선 어떻게 사용할까?

아래같은 형태가 자주 나온다.

In [ ]:
conv1 = nn.Conv2d( # 공간 유지
    in_channels=3,
    out_channels=64,
    kernel_size=3,
    padding=1,
    stride=1
)

In [ ]:
conv2 = nn.Conv2d( # 공간 축소 224 X 224 -> 112 X 112
    in_channels=64,
    out_channels=128,
    kernel_size=3,
    padding=1,
    stride=2
)

CNN을 깊게 만들면서 흔히 H x W는 점점 감소시키고, 채널 수는 증가시킨다.

예를 들어서
```text
3 × 224 × 224
        ↓
64 × 224 × 224
        ↓
128 × 112 × 112
        ↓
256 × 56 × 56
```
같은 구조를 만들 수 있다. Stride를 크게 하면 계산량을 줄이면서 공간 해상도를 낮출 수 있다는 것이 핵심이다.

## 14. 이해하고 가야하는 계산

In [ ]:
nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    padding=1,
    stride=2
)

입력이 [32, 3, 64, 64]라면 먼저 H, W만 계산한다.

$$
\left\lfloor \frac{64 + 2(1) - 3}{2} \right\rfloor + 1\\ = \lfloor31.5\rfloor + 1\\ = 32
$$

그래서 출력은 [32, 16, 32, 32] 이렇다.

H, W는 줄고 channel은 늘었다.

CNN 구조를 읽기 위해 이정도는 알아야한다.

## 15. 오늘의 정리

- Convolution을 적용하면 기본적으로 feature map의 높이와 너비가 줄어든다.
- padding은 입력의 가장자리에 값을 추가해서 출력 크기가 너무 빨리 감소하는 것을 막는다.
- 가장 일반적인 padding은 zero padding이다.
- 3 × 3 kernel + padding=1 + stride=1이면 보통 입력과 출력의 높이·너비가 같다.
- CNN에서 3×3, 5×5, 7×7 같은 홀수 커널을 많이 사용하는 이유 중 하나는 대칭 padding이 쉽기 때문이다.
- stride는 커널이 이동하는 간격이다.
- stride=1이면 한 칸씩 이동하고 stride=2이면 두 칸씩 이동한다.
- stride가 커질수록 출력 feature map은 작아진다.
- padding은 주로 크기 보존, stride는 주로 크기 축소라고 이해하면 된다.